## Statistical data analysis. Midterm 

**(!) <span style="color:blue"> Add your name to the file name instead of [Name]</span>**</br>
Example: Stat_analysis_25-26_midterm_Ivanov Ivan.ipynb

Submit your notebook to Smart LMS https://edu.hse.ru/course/view.php?id=268179 (access at the end of assessment only)

Unlocked web sites to visit:</br>
* https://docs.scipy.org/doc/scipy/tutorial/index.html
* https://docs.python.org/3/library/random.html
* https://docs.python.org/3/library/
* https://numpy.org/
* https://pandas.pydata.org/docs/user_guide/index.html
* https://matplotlib.org/
* https://plotly.com/python/

In [ ]:
import numpy as np
import random
import plotly.graph_objects as go

from scipy.stats import uniform

**Question 1 [10 pts]**. Compute expectation $E(A(r)\cdot e^r),\, r\sim U[0.15, 0.2]$ from Problem 1 via Monte Carlo simulations.

Use $N=25\,000$ simulations, output computed approximation of expectation.

In [ ]:
N = 25_000
rs = uniform.rvs(loc = 0.15, scale = 0.05, size = N)

A = []
for r in rs:
    if r >= 0.15 and r <= 0.16:
        A.append(10)
        continue
    if r > 0.16 and r <= 0.18:
        A.append(15)
        continue
    if r > 0.18 and r <= 0.2:
        A.append(20)
        continue

In [ ]:
A_np = np.array(A)
A_np*np.exp(rs)
amount = np.mean(A_np*np.exp(rs))
print(amount)

If you computed theoretical expectation output absolute difference $|\overline{A(r)\cdot e^r} - E(A(r)\cdot e^r)|.$

In [ ]:
theor_mean = 100*(4*np.exp(0.2)-np.exp(0.18)-np.exp(0.16)-2*np.exp(0.15))
print(theor_mean, abs(theor_mean-amount))

**Question 2 [23 pts]** Implement the process $(C_n)_{n\ge 0}$ defined at Problem 2b) by simulating trajectories. Complete a function below:

In [ ]:
def modified_binom_simul(C0, u, d, p, A, B, time, n_paths):
    paths = []    
    for _ in range(n_paths):
        path = [C0]
        for i in range(1, time + 1):
            if path[-1] > A and path[-1] < B:
                val = random.choices([u, d], weights = [p, 1-p])[0]
                path.append(path[-1]*val)
                continue
            if path[-1] < A:
                path.append(path[-1]*u)
                continue
            if path[-1] > B:
                path.append(path[-1]*d)
                continue
        paths.append(np.array(path))

    return np.array(paths)

Simulate $N=10\,000$ trajectories of $(C_n)_{n\le 30}$, parameters $u=1.04,\,d = 0.96,\,p=0.5,$ initial state $C_0=100,$ boundaries $A=90,\, B=120$.</br>
Approximate and output probability $p_c = P(95<C_{30}<110).$ 

In [ ]:
C0 = 100
u = 1.04
d = 0.96
p = 0.5
A = 90
B = 120

n = 30
N = 10_000

paths = modified_binom_simul(C0, u, d, p, A, B, n, N)

Cn = paths[:,-1]
prob_c = sum((Cn>95) & (Cn<110))/N
print(prob_c)

Simulate $N=10\,000$ trajectories of Binomial model $(S_n)_{n\le 30},$ parameters $u=1.04,\,d=0.96,\,p=0.5$ (same as above), initial state $S_0=100$.</br>
Approximate and output probability $p_b = P(95<S_{30}<110).$ 

In [ ]:
# Call, but DO NOT MODIFY THIS FUNCTION
def binom_simulations(S0, u, d, p, time, n_paths):
    
    trajectories = []    
    for _ in range(n_paths):
        xs = random.choices([u, d], weights = [p, 1-p], k = time)
        xs_initial = [1] + xs
        prices = S0*np.cumprod(xs_initial)
        trajectories.append(prices)
        
    return np.array(trajectories)    

In [ ]:
S0 = 100
paths_binom = binom_simulations(S0, u, d, p, n, N)
Sn = paths_binom[:,-1]
prob_b = sum((Sn>95) & (Sn<110))/N
print(prob_b)

Output ratio $l = \displaystyle\frac{p_c}{p_b}$

In [ ]:
l = prob_c/prob_b
print(l)

**Question 3 [17 pts]** Simulation of the process 
$$
\large{X_t = t e^{W_t-2t},\, t\ge 0,}
$$ 
where $(W_t)_{t\ge 0}$ is Brownian motion.

In [ ]:
# Call, but DO NOT MODIFY THIS FUNCTION
def bm_simulations(n_paths, granularity, max_time):
    n_points = granularity * max_time
    
    paths = []
    for _ in range(n_paths):
        xs = [0] + random.choices([-1, 1], weights = [0.5, 0.5], k = n_points)
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
    
    return np.array(paths)

1) Use function bm_simulations() to simulate $N=5$ paths of Brownian motion $W_t,\, 0\le t \le 2$. Use $granulatiry = 1000$. Modify obtained paths and get paths of $X_t,\, 0\le t \le 2.$ </br>
Output these paths in form of scatterplots.

In [ ]:
N = 5
n = 1000
T = 2

t = np.linspace(0, T, n*T+1) 
paths = bm_simulations(N,n,T)
paths_x = [t*np.exp(path - 2*t) for path in paths]

In [ ]:
plot = go.Figure()

for path_x in paths_x:
    graph = go.Scatter(x = t, y = path_x)
    plot.add_trace(graph)
    
plot.show()

2)  Use function bm_simulations() to simulate $N=10\,000$ paths of Brownian motion $W_t,\, 0\le t \le 2$. Use $granulatiry = 100$. </br>

In [ ]:
N = 10_000
n = 100
T = 2

paths = bm_simulations(N,n,T)

2.1) Obtain $X_{1.5}=1.5 e^{W_{1.5}-3}$ and output sample mean of $X_{1.5}$. If you solved problem 3c) output absolute difference $\left|sample\_mean(X_{1.5}) - E(X_{1.5})\right|.$

In [ ]:
x1 = 1.5*np.exp(paths[:,150]-3)
mean = np.mean(x1)
theor_mean = 1.5*np.exp(-9/4)
print(mean, abs(mean-theor_mean))

2.2) Obtain $X_{0.5}=0.5e^{W_{0.5}-1}$ and output sample covariance between $X_{1.5}$ and $X_{0.5}$. If you solved 3d) output absolute difference $|sample\_cov(X_{1.5}, X_{0.5})-cov(X_{1.5}, X_{0.5})|$.

In [ ]:
def theor_cov(t,s):
    return t*s*np.exp(-1.5*(t+s))*(np.exp(s)-1)

x2 = 0.5*np.exp(paths[:,50]-1)
sample_cov = np.cov(x1, x2)[0][1]

print(sample_cov, abs(sample_cov - theor_cov(1.5, 0.5)))